# TOAP KV-Bridge Benchmark (Colab)

Real KV-bridge experiment: **recompute** (re-prefill prefix+query) vs **KV-bridge** (reuse the
transferred prefix KV and prefill only the query). Per shared-context length we measure:

1. **prefill latency** — recompute vs bridge (the compute win),
2. **transfer cost** — KV serialize/deserialize time (reported separately, never hidden),
3. **KV byte size vs text size** — the honest storage/bandwidth cost,
4. **correctness** — greedy tokens must match the recompute baseline exactly (losslessness).

Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU), then **Run all**. A model matrix
spanning MHA (GPT-2), Pythia, GQA (Qwen2.5), and a 4-bit 7B runs with per-model error isolation —
if one model fails or OOMs, the rest still complete. At the end, `kv_bench_all.json` is produced;
send it back to fold into the TOAP paper.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> T4 GPU, then Run all.'

In [ ]:
# Dependencies. bitsandbytes + accelerate enable the optional 4-bit 7B model.
!pip -q install -U transformers accelerate bitsandbytes
!git clone https://github.com/parnish007/TOAP.git
%cd TOAP/benchmark/kv_bridge
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# Smoke test on the smallest model first — confirms the pipeline works before the big runs.
# Asserts non-empty rows AND token-lossless match, so a version/API bug is caught in seconds
# (not after the full matrix). If this fails, STOP and report the printed error.
import json, subprocess
subprocess.run(['python', 'kv_bench.py', '--model', 'sshleifer/tiny-gpt2',
                '--prefix-tokens', '64', '128', '--new-tokens', '8', '--repeats', '5',
                '--out', 'smoke.json'], check=True)
_sm = json.load(open('smoke.json'))
assert _sm['rows'], 'SMOKE FAILED: no rows produced (a cache/API bug — do not run the matrix yet).'
assert all(r['outputs_match'] for r in _sm['rows']), 'SMOKE FAILED: KV-bridge not token-lossless.'
print('smoke OK:', [(r['prefix_tokens'], round(r['speedup_resident'], 2), r['outputs_match'])
                    for r in _sm['rows']])

In [ ]:
# Model matrix — publication config. CRASH-SAFE + RESUMABLE: kv_bench_all.json is rewritten after
# EVERY model (so a power loss / disconnect never loses completed work), and a model whose per-model
# res_*.json already exists this session is loaded from disk instead of re-run. So if you got cut off,
# just Run-all again: finished models are skipped instantly and it resumes where it stopped.
import json, subprocess, os, time

MATRIX = [
    ('gpt2',                       ['--model', 'gpt2', '--prefix-tokens', '128', '256', '512', '768', '960']),
    ('gpt2-large',                 ['--model', 'gpt2-large', '--prefix-tokens', '128', '256', '512', '768', '960']),
    ('EleutherAI/pythia-410m',     ['--model', 'EleutherAI/pythia-410m', '--prefix-tokens', '128', '256', '512', '1024', '2000']),
    ('EleutherAI/pythia-1.4b',     ['--model', 'EleutherAI/pythia-1.4b', '--prefix-tokens', '128', '256', '512', '1024', '2000']),
    ('Qwen/Qwen2.5-0.5B-Instruct', ['--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--prefix-tokens', '256', '512', '1024', '2048', '4096', '8192']),
    ('Qwen/Qwen2.5-1.5B-Instruct', ['--model', 'Qwen/Qwen2.5-1.5B-Instruct', '--prefix-tokens', '256', '512', '1024', '2048', '4096', '8192']),
    # Mistral-7B-4bit is the slow long pole and is OPTIONAL: the 6 models above already span MHA, NeoX,
    # and GQA. If it OOMs / times out / is interrupted, the other six are already saved and complete.
    ('mistralai/Mistral-7B-Instruct-v0.3', ['--model', 'mistralai/Mistral-7B-Instruct-v0.3', '--load-in-4bit', '--prefix-tokens', '256', '512', '1024', '2048']),
]
COMMON = ['--new-tokens', '8', '--repeats', '15']

def save(results):
    with open('kv_bench_all.json', 'w') as f:
        json.dump({'runs': results}, f, indent=2)

results = []
for name, args in MATRIX:
    out = f"res_{name.replace('/', '__')}.json"
    # resume: reuse a completed per-model result from this session
    if os.path.exists(out):
        try:
            results.append(json.load(open(out)))
            print(f'[cached] {name} (already done)')
            save(results)
            continue
        except Exception:
            pass
    print(f'\n===== {name} =====', flush=True)
    t0 = time.time()
    try:
        subprocess.run(['python', 'kv_bench.py', *args, *COMMON, '--out', out],
                       check=True, timeout=2400)
        if os.path.exists(out):
            results.append(json.load(open(out)))
            print(f'[ok] {name} in {time.time()-t0:.0f}s')
    except subprocess.TimeoutExpired:
        print(f'[skip] {name}: timed out')
    except subprocess.CalledProcessError as e:
        print(f'[skip] {name}: exited {e.returncode}')
    except Exception as e:
        print(f'[skip] {name}: {type(e).__name__}: {e}')
    save(results)                     # <-- incremental save after EVERY model (crash-safe)
    torch.cuda.empty_cache()

print(f'\n[written] kv_bench_all.json with {len(results)} model runs')

In [ ]:
# Quick summary table across all models that completed.
import json
data = json.load(open('kv_bench_all.json'))
print(f"{'model':32} {'dtype':9} {'prefix':>6} {'kv/text':>8} {'sp_resident':>11} {'sp_xnode':>9} {'match':>6}")
print('-' * 90)
for run in data['runs']:
    for r in run['rows']:
        print(f"{run['model'][:32]:32} {run['dtype'][:9]:9} {r['prefix_tokens']:>6} "
              f"{(r['kv_vs_text_ratio'] or 0):>6.0f}x {r['speedup_resident']:>10.2f}x "
              f"{r['speedup_crossnode']:>8.2f}x {str(r['outputs_match']):>6}")
print('\nsp_resident = co-located compute win (KV already on GPU); sp_xnode = includes transfer cost.')

In [ ]:
# Publication figures: (1) prefill speedup vs prefix length per model, (2) KV/text size ratio.
import json
import matplotlib.pyplot as plt
data = json.load(open('kv_bench_all.json'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
for run in data['runs']:
    rows = sorted(run['rows'], key=lambda r: r['prefix_tokens'])
    if not rows:
        continue
    xs = [r['prefix_tokens'] for r in rows]
    label = run['model'].split('/')[-1]
    ax1.plot(xs, [r['speedup_resident'] for r in rows], marker='o', label=label)
    ax2.plot(xs, [r['kv_vs_text_ratio'] for r in rows], marker='s', label=label)

ax1.axhline(1.0, ls='--', color='gray', lw=0.8)
ax1.set_xlabel('shared prefix length (tokens)'); ax1.set_ylabel('prefill speedup (resident, x)')
ax1.set_title('KV-bridge compute win vs prefix length\n(>1 = faster than recompute)')
ax1.set_xscale('log', base=2); ax1.legend(fontsize=8)

ax2.set_xlabel('shared prefix length (tokens)'); ax2.set_ylabel('KV size / text size (x)')
ax2.set_title('Honest cost: KV cache vs the text it replaces')
ax2.set_xscale('log', base=2); ax2.set_yscale('log'); ax2.legend(fontsize=8)

fig.tight_layout()
fig.savefig('kv_bridge_figure.png', dpi=150)
plt.show()
print('saved kv_bridge_figure.png')

In [ ]:
from google.colab import files
files.download('kv_bench_all.json')   # send this back to fold into the paper
files.download('kv_bridge_figure.png')